# Codestral Vulkan trên Google Colab (A–Z)

Chọn **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từ trên xuống. Notebook cài Nim 2.2.10, clone repo kèm submodule, cài Vulkan, tải tokenizer/model/adapter và chạy inference bằng Vulkan. Tokenizer Codestral chạy thuần Nim; không cần cài Nimpy, SentencePiece hay gói tokenizer Python.

Repo mặc định vẫn `auto`; notebook truyền `BYBY_INFER_BACKEND=vulkan` riêng cho lần chạy này. Weights mmap từ file; merge LoRA tắt để tiết kiệm RAM/VRAM. Sửa `PROMPT` và `MAX_NEW_TOKENS` ở cell inference.


In [1]:
# 1) Kiểm tra runtime GPU và cài các gói hệ thống, gồm Vulkan SDK/runtime và shader compiler.
import os, subprocess, sys
from pathlib import Path

assert Path('/content').exists(), 'Notebook này cần chạy trên Google Colab.'
try:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'], text=True))
except Exception as e:
    raise RuntimeError('Chưa có GPU NVIDIA. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.') from e

subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','git','curl','ca-certificates','xz-utils','pkg-config','time','libvulkan1','libvulkan-dev','vulkan-tools','glslc','libegl1','libglx0','libx11-6','libxext6','libx11-xcb1','libxcb1','libxcb-dri3-0','libxcb-present0','libxfixes3','libxrandr2','libdrm2','libexpat1','libwayland-client0'], check=True)


Tesla T4, 580.82.07, 15360 MiB



CompletedProcess(args=['apt-get', 'install', '-y', '-qq', 'build-essential', 'git', 'curl', 'ca-certificates', 'xz-utils', 'pkg-config', 'time', 'libvulkan1', 'libvulkan-dev', 'vulkan-tools', 'glslc', 'libegl1', 'libglx0', 'libx11-6', 'libxext6', 'libx11-xcb1', 'libxcb1', 'libxcb-dri3-0', 'libxcb-present0', 'libxfixes3', 'libxrandr2', 'libdrm2', 'libexpat1', 'libwayland-client0'], returncode=0)

In [2]:
# 2) Cài Nim 2.2.10 chính chủ và xác minh checksum (tokenizer thuần Nim).
import hashlib, urllib.request, tarfile, os, subprocess
from pathlib import Path

NIM_VERSION = '2.2.10'
TOOLS = Path('/content/tools'); TOOLS.mkdir(exist_ok=True)
archive = TOOLS / f'nim-{NIM_VERSION}-linux_x64.tar.xz'
url = f'https://nim-lang.org/download/nim-{NIM_VERSION}-linux_x64.tar.xz'

# Tải file với User-Agent giả lập trình duyệt để tránh lỗi 403
if not archive.exists():
    req = urllib.request.Request(
        url,
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    )
    with urllib.request.urlopen(req) as response, open(archive, 'wb') as out_file:
        out_file.write(response.read())

# Xác minh checksum
expected = '0a3a38752e97e9d44aa479b3a7b37336dfe0176daf22ee5b5218ad0991ecd211'
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
assert actual == expected, f'Nim checksum mismatch: {actual}'

# Giải nén
with tarfile.open(archive, 'r:xz') as tf:
    tf.extractall(TOOLS, filter='data')

# Cấu hình PATH
NIM_HOME = TOOLS / f'nim-{NIM_VERSION}'
os.environ['PATH'] = f"{NIM_HOME}/bin:" + os.environ['PATH']

# Kiểm tra phiên bản
subprocess.run(['nim', '--version'], check=True)


CompletedProcess(args=['nim', '--version'], returncode=0)

In [3]:
# 3) Clone repository bản main mới nhất và tải submodule đã ghim trong repo.
REPO = Path('/content/nimformers-framework')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/bobbyshop-vui/nimformers-framework.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True)
subprocess.run(['git','-C',str(REPO),'submodule','sync','--recursive'], check=True)
subprocess.run(['git','-C',str(REPO),'submodule','update','--init','--recursive'], check=True)
os.chdir(REPO)
print('Repo:', REPO)


Repo: /content/nimformers-framework


In [4]:
# 4) Cài Vulkan + kiểm tra ICD NVIDIA và xác nhận dGPU (không chấp nhận lavapipe/CPU)
import subprocess, os, re
from pathlib import Path

# --- Bước 1: Cài vulkan-tools ---
print("[1/4] Cài vulkan-tools...")
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'vulkan-tools'], check=True)

# --- Bước 2: Xác định phiên bản driver NVIDIA và cài libnvidia-gl tương ứng ---
print("[2/4] Phát hiện phiên bản driver NVIDIA...")
smi = subprocess.check_output(['nvidia-smi'], text=True)
m = re.search(r'Driver Version:\s*(\d+)\.', smi)
assert m, "Không đọc được Driver Version từ nvidia-smi"
driver_major = m.group(1)
print(f"    -> Driver major: {driver_major}")

pkg = f'libnvidia-gl-{driver_major}'
print(f"[2/4] Cài {pkg}...")
subprocess.run(['apt-get', 'install', '-y', '-qq', pkg], check=True)

# --- Bước 3: Đảm bảo file ICD NVIDIA có thể được loader tìm thấy ---
print("[3/4] Cấu hình Vulkan ICD...")
Path('/usr/share/vulkan/icd.d').mkdir(parents=True, exist_ok=True)
candidates = [
    '/etc/vulkan/icd.d/nvidia_icd.json',
    '/usr/share/vulkan/icd.d/nvidia_icd.json',
    f'/usr/lib/x86_64-linux-gnu/libnvidia-gl-{driver_major}/nvidia_icd.json',
]
src = next((c for c in candidates if Path(c).exists()), None)
if src and src != '/usr/share/vulkan/icd.d/nvidia_icd.json':
    subprocess.run(['ln', '-sf', src, '/usr/share/vulkan/icd.d/nvidia_icd.json'], check=True)
    print(f"    -> symlink: {src} -> /usr/share/vulkan/icd.d/nvidia_icd.json")
else:
    print(f"    -> ICD có sẵn: {src}")

# --- Bước 4: Chạy kiểm tra ---
print("[4/4] Kiểm tra Vulkan...")
subprocess.run(['vulkaninfo', '--summary'], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
summary = subprocess.check_output(['vulkaninfo', '--summary'],
                                   text=True, stderr=subprocess.STDOUT)
print(''.join(line for line in summary.splitlines()
              if any(k in line.lower() for k in ('device name', 'device type', 'driver name'))))

assert 'NVIDIA' in summary.upper(), \
    'Không thấy NVIDIA Vulkan ICD. Runtime GPU hiện tại không expose Vulkan graphics.'
assert 'DISCRETE_GPU' in summary.upper(), \
    'Không tìm thấy NVIDIA dGPU qua Vulkan; dừng để tránh fallback CPU/iGPU.'

print("✅ Vulkan OK: NVIDIA dGPU đã được expose.")

[1/4] Cài vulkan-tools...
[2/4] Phát hiện phiên bản driver NVIDIA...
    -> Driver major: 580
[2/4] Cài libnvidia-gl-580...
[3/4] Cấu hình Vulkan ICD...
    -> ICD có sẵn: /usr/share/vulkan/icd.d/nvidia_icd.json
[4/4] Kiểm tra Vulkan...

✅ Vulkan OK: NVIDIA dGPU đã được expose.


In [5]:
# 5) Tải Codestral tokenizer/base FP4-KM và adapter. Hugging Face tự resume nếu cell bị ngắt.
subprocess.run([sys.executable,'-m','pip','-q','install','huggingface_hub'], check=True)
from huggingface_hub import hf_hub_download, model_info

MODELS = REPO / 'test/codestral/models'
HF_DIR = MODELS / 'hf'; HF_DIR.mkdir(parents=True, exist_ok=True)
BASE_REPO = 'bobbydeveloper2014/codestral-v0.1-7b-nimq'
BASE_REV = '148dd26885b5eeef03476683f05f0a4546cd75ba'
for name in ['codestral_mamba_only.nimq','config.json','generation_config.json','params.json','special_tokens_map.json','tokenizer.json','tokenizer.model','tokenizer.model.v3','tokenizer_config.json']:
    hf_hub_download(BASE_REPO, filename=name, revision=BASE_REV, local_dir=str(HF_DIR if name.endswith('.json') or name.startswith('tokenizer.') else MODELS))
ADAPTER_REPO = 'bobbydeveloper2014/codestral-v0.1-nimq-qlora-linearattention'
ADAPTER_REV = model_info(ADAPTER_REPO).sha
adapter = hf_hub_download(ADAPTER_REPO, filename='jamba_parallel_lora.bin', revision=ADAPTER_REV, local_dir=str(MODELS))
Path(str(adapter)+'.hf_tokenizer_dir').write_text(str(HF_DIR.resolve()))
print('Base + tokenizer và adapter đã tải. Adapter revision:', ADAPTER_REV)


codestral_mamba_only.nimq: reconstructing file:   0%|          |  0.00B / 4.34GB            

codestral_mamba_only.nimq: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

params.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer.model.v3: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

jamba_parallel_lora.bin: reconstructing file:   0%|          |  0.00B / 1.65GB            

jamba_parallel_lora.bin: downloading bytes:           |  0.00B            

Base + tokenizer và adapter đã tải. Adapter revision: 7d6efea2844839808ae4cbfa2b5f822c140054bf


In [6]:
# 6) Biên dịch shader Vulkan và binary inference.
import os
os.chdir(str(REPO))

!make shaders
!make build-infer-lora

[shaders] xong.
nim check -d:release --hints:off --mm:arc --opt:speed --path:. --path:test/codestral --path:test/deepseek-inference --path:test/spectron --path:kernelfortest kernelfortest/fp4_gemv_gpu.nim
/content/nimformers-framework/vendor/bybylang/backends/metal/metal_backend.nim(2, 12) Warning: imported and not used: 'os' [UnusedImport]
/content/nimformers-framework/vendor/bybylang/gpubackend.nim(825, 5) Warning: unreachable else, all cases are already covered [UnreachableElse]
nim check -d:release --hints:off --mm:arc --opt:speed --path:. --path:test/codestral --path:test/deepseek-inference --path:test/spectron --path:kernelfortest -d:codestralInference test/codestral/infer_lora.nim
/content/nimformers-framework/vendor/bybylang/backends/metal/metal_backend.nim(2, 12) Warning: imported and not used: 'os' [UnusedImport]
/content/nimformers-framework/vendor/bybylang/gpubackend.nim(825, 5) Warning: unreachable else, all cases are already covered [UnreachableElse]
/content/nimformers-f

## Inference

Model mmap trực tiếp từ `.nimq`; LoRA không merge vào bản sao dense. `BYBY_VULKAN_DEVICE_LOCAL=auto` để backend chỉ đặt weights ở VRAM private khi Vulkan nhận diện dGPU NVIDIA. T4 dùng Tensor Core FP16 nếu kernel/driver hỗ trợ; FP4 là định dạng lưu trữ, không giả định T4 có phép tính FP4 native.


In [7]:
# 7) Chạy inference.
import os
from pathlib import Path

PROMPT = '[INST] Write a Python function for Fibonacci. [/INST]'
MAX_NEW_TOKENS = 99999999
model = MODELS / 'jamba_parallel_lora.bin'
base = MODELS / 'codestral_mamba_only.nimq'
assert model.exists() and base.exists() and (HF_DIR/'tokenizer.json').exists()

os.chdir(str(REPO))

print('Backend Vulkan | mmap | max tokens =', MAX_NEW_TOKENS)
print('model =', model)

!BYBY_INFER_BACKEND=vulkan \
 BYBY_PROGRESS=0 \
 BYBY_MODEL_MMAP=1 \
 BYBY_MERGE_LORA=0 \
 BYBY_VULKAN_DEVICE_LOCAL=auto \
 BYBY_VULKAN_TENSOR_CORES=1 \
 BYBY_GEMV_TG=max \
 BYBY_MAX_NEW_TOKENS={MAX_NEW_TOKENS} \
 BYBY_PREFILL_RESIDENT=1 \
 ./infer_lora_bin "{model}" "{PROMPT}" 0.0 "{HF_DIR}" "{base}"

Backend Vulkan | mmap | max tokens = 99999999
model = /content/nimformers-framework/test/codestral/models/jamba_parallel_lora.bin
[infer] GEMV_TG=max autotune=1 debug_sampling=0; prefill GEMM=auto; prefill scan=auto; FP4 2col=1; FP4 kernel=auto; resident prefill=1; decode=native (no MPS)
== [Framework] Backend đã chọn: vulkan ==
[infer] Phát hiện NIMBIN4 (QLoRA 4-bit) — newBlock giữ nén, dequant on-the-fly lúc generate()...
[infer] checkpoint ghi preset norm luc train: rms-fin-gn
[infer] checkpoint ghi Mamba norm implementation: mamba-ssm-postgate-grouped-v1
[infer] Pattern block (64 block): RRRRRRRGRRRRRRRRRRRRRRRGRRRRRRRRRRRRRRRGRRRRRRRRRRRRRRRGRRRRRRRR
[infer] Thanh phan: R(Mamba2Real)=60 P(Mamba2Real+Linear++ sum)=0 G(Mamba2Real+gated Linear++ fuseGate)=4 M(Mamba)=0 A(LinearPlusAttention)=0 S(Spectron)=0
[hf_tokenizer] pure Nim BPE, vocab=32768, normalizer=identity, bos=1 eos=2 unk=0 stopIds=@[2, 3, 4]
[load_codestral] (quant-resident) Doc /content/nimformers-framework/test/codestr

### Tài liệu

- [Nim 2.2.10](https://nim-lang.org/download/)
- [Google Colab FAQ: GPU runtime](https://research.google.com/colaboratory/faq.html)
- [NVIDIA Container Toolkit: graphics capability](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/docker-specialized.html)
